In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib.pyplot import imshow
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
import os
import keras_cv
from keras import layers
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, f1_score, confusion_matrix

In [ ]:
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Embedding
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer

In [ ]:
IMG_H, IMG_W = 64, 64   
BATCH = 32

train_dir = r"D:\friday_20_dataset\train"
val_dir = r"D:\friday_20_dataset\val"
test_dir = r"D:\friday_20_dataset\test"

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    labels="inferred",
    label_mode="binary",
    color_mode="rgb",
    image_size=(IMG_H, IMG_W),
    batch_size=BATCH,
    shuffle=True
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    val_dir,
    labels="inferred",
    label_mode="binary",
    color_mode="rgb",
    image_size=(IMG_H, IMG_W),
    batch_size=BATCH,
    shuffle=False
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    labels="inferred",
    label_mode="binary",
    color_mode="rgb",
    image_size=(IMG_H, IMG_W),
    batch_size=BATCH,
    shuffle=False
)


def preprocess(img, label):
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

train_ds = train_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)


In [ ]:
#patching & embedding
class PatchEmbedding(tf.keras.layers.Layer):
    def __init__(self, patch_size, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.proj = Dense(embed_dim)

    def call(self, images):
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID'
        )
        batch = tf.shape(images)[0]
        patch_dim = patches.shape[-1]
        patches = tf.reshape(patches, [batch, -1, patch_dim])
        return self.proj(patches)


In [ ]:
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, mlp_dim):
        super().__init__()
        self.norm1 = LayerNormalization(epsilon=1e-6)
        self.att = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.norm2 = LayerNormalization(epsilon=1e-6)
        self.mlp = tf.keras.Sequential([
            Dense(mlp_dim, activation="gelu"),
            Dense(embed_dim),
        ])

    def call(self, x):
        h = self.att(x, x)
        x = self.norm1(x + h)
        h = self.mlp(x)
        return self.norm2(x + h)


In [ ]:
class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, num_patches, embed_dim):
        super().__init__()
        self.pos_emb = Embedding(num_patches, embed_dim)

    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[1], delta=1)
        pos_embeddings = self.pos_emb(positions)
        return x + pos_embeddings


In [ ]:
class TokenPooling(Layer):
    def call(self, x):
        return tf.reduce_mean(x, axis=1)

In [ ]:
IMG_H, IMG_W = 64, 64
PATCH = 8

NUM_PATCHES = (IMG_H // PATCH) * (IMG_W // PATCH)
EMBED_DIM = 128
NUM_HEADS = 4
MLP_DIM = 256
NUM_BLOCKS = 4

def build_vit():
    inp = Input((IMG_H, IMG_W, 3))

    x = PatchEmbedding(PATCH, EMBED_DIM)(inp)
    x = PositionalEmbedding(NUM_PATCHES, EMBED_DIM)(x)

    for _ in range(NUM_BLOCKS):
        x = TransformerBlock(EMBED_DIM, NUM_HEADS, MLP_DIM)(x)

    x = LayerNormalization()(x)
    x = TokenPooling()(x)    
    out = Dense(1, activation="sigmoid")(x)

    return Model(inp, out)


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

callback = EarlyStopping(
    monitor='val_loss',           
    patience=5,                   
    restore_best_weights=True,    
    verbose=1                     
)

In [ ]:
vit = build_vit()

vit.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

vit.fit(train_ds, validation_data=val_ds, callbacks=[callback],epochs=20)


In [ ]:
test_loss, test_acc = vit.evaluate(test_ds)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
import seaborn as sns

y_pred_proba = vit.predict(test_ds)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()


y_true = np.concatenate([y for x, y in test_ds], axis=0)

print(classification_report(y_true, y_pred, target_names=['Benign', 'DDoS']))

print("Confusion Matrix..")
cm = confusion_matrix(y_true, y_pred)
print(cm)


from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

print(f"AUC-ROC:   {roc_auc_score(y_true, y_pred_proba):.4f}")
print(f"F1-Score:  {f1_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred):.4f}")

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Benign', 'DDoS'],
            yticklabels=['Benign', 'DDoS'])
plt.title('Confusion Matrix - Vision Transformer')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()